In [ ]:
param_intermediate_location = "/home/jovyan/Cloud Storage/naa-vre-veluwe-flm-minio"
conf_tmp_data = "/tmp/data"
conf_data_input_location <- "/home/jovyan/Cloud Storage/naa-vre-public/vl-veluwe-forest-model/"
output_base_path <- file.path(param_intermediate_location, paste0("workflow_run_", "20260825-1104", "/outputs"))

# Data extraction and preparation
Extract outputs from all parameter runs and aggregate it into a single csv file for analysis. 

In [ ]:
# Extract biomass-succession log outputs from all runs
# ====================================================
# Get list of all run folders (001-729)
run_folders <- list.dirs(path = output_base_path, full.names = TRUE, recursive = FALSE)
run_numbers <- basename(run_folders)

total_runs <- length(run_numbers)

# Unzip each output.zip file
for (i in seq_along(run_numbers)) {
  run_num <- run_numbers[i]
  tar_file <- file.path(output_base_path, run_num, "outputs.tar.gz")
  output_dir <- file.path(output_base_path, run_num)
  
  # Check if already unzipped - if there are files besides outputs.zip, skip
  existing_files <- list.files(output_dir)
  if (any(existing_files != "outputs.tar.gz" & existing_files != "Metadata")) {
    cat(sprintf("[%d/%d] Skipping %s - already unzipped\n", i, total_runs, run_num))
    next
  }
  
  # Check if zip file exists
  if (!file.exists(tar_file)) {
    cat(sprintf("[%d/%d] ERROR: %s - zip file not found\n", i, total_runs, run_num))
    next
  }
  
  cat(sprintf("[%d/%d] Unzipping %s...\n", i, total_runs, run_num))
  untar(tar_file, 
        exdir = output_dir)
}

In [ ]:
# Create empty list to store dataframes
all_data <- list()

# Loop through each run folder
for (run_num in run_numbers) {

  # Build path to the biomass log file
  biomass_file <- file.path(output_base_path, run_num, "Biomass-succession-log.csv")
  
    # Read the CSV
    data <- readr::read_csv(biomass_file, show_col_types = FALSE)
    
    # Select only the columns you care about and add run column
    data_clean <- data |>
      dplyr::select(Time, AvgLiveB, AvgAG_NPP) |>
      dplyr::mutate(run = run_num)
    
    # Add to list
    all_data[[run_num]] <- data_clean
}

# Combine all dataframes into one
combined_data <- dplyr::bind_rows(all_data)

readr::write_csv(combined_data, paste0(output_base_path, "_combined_biomass_succession_data.csv"))

## Visualisation
* Exploration of parameter sweep model runs
* Visualisation of all model growth trajectories

In [ ]:
# Prepare data to highlight parameter differences
# ========================================================

parameters <- tibble::tribble(
  ~"parameter",        ~"species",            ~"initial",   ~"min",   ~"max",    ~"calibrated",
  # --- ANPPMax ---
  "ANPPMax",         "betula",                 1130,         548,      1520,      548,
  "ANPPMax",         "pinussylvestris",        712,          551,      1375,      551,
  "ANPPMax",         "quercus",                1130,         548,      1392,      548,
  "ANPPMax",         "pseudotsugamenziesii",   712,          551,      1865,      551,
  # --- BiomassMax ---
  "BiomassMax",      "betula",                 17740,        6270,     15150,     27300,
  "BiomassMax",      "pinussylvestris",        17740,        6540,     14390,     27300,
  "BiomassMax",      "quercus",                17740,        6270,     15150,     27300,
  "BiomassMax",      "pseudotsugamenziesii",   17740,        6540,     14390,     27300,
  # --- GrowthCurve ---
  "GrowthCurve",     "betula",                 0.74,         0.2,      0.87,      0.2,
  "GrowthCurve",     "pinussylvestris",        0.74,         0.62,     1.0,       0.62,
  "GrowthCurve",     "quercus",                0.74,         0.4,      0.92,      0.4,
  "GrowthCurve",     "pseudotsugamenziesii",   0.74,         0.2,      0.87,      0.2,
  # --- MortalityCurve ---
  "MortalityCurve",  "betula",                 12,            5,        15,        12,
  "MortalityCurve",  "pinussylvestris",        12,            9,        25,        12,
  "MortalityCurve",  "quercus",                12,            9,        25,        12,
  "MortalityCurve",  "pseudotsugamenziesii",   12,            9,        25,        12,
  # --- Longevity ---
  "Longevity",       "betula",                 220,          90,       150,       150,
  "Longevity",       "pinussylvestris",        1000,         400,      750,       750,
  "Longevity",       "quercus",                1400,         500,      1000,      1000,
  "Longevity",       "pseudotsugamenziesii",   1400,         500,      1000,      1000,
  # --- ProbEstablish ---
  "ProbEstablish",   "betula",                 1.0,          0.3,      0.5,       0.3,
  "ProbEstablish",   "pinussylvestris",        1.0,          0.5,      0.5,       0.5,
  "ProbEstablish",   "quercus",                0.1,          0.1,      0.3,       0.1,
  "ProbEstablish",   "pseudotsugamenziesii",   1.0,          0.3,      0.5,       0.3
)

# Generate parameter grid
groups <- c("initial", "min", "max")
param_grid <- expand.grid(
  ANPPMax = groups,
  BiomassMax = groups,
  GrowthCurve = groups,
  MortalityCurve = groups,
  Longevity = groups,
  ProbEstablish = groups,
  stringsAsFactors = FALSE
) |>
  dplyr::mutate(
    run = sprintf("%03d", dplyr::row_number())
  )
# ========== CONVERT TO LONG FORMAT FOR GGPLOT ==========
# This is the KEY step for effective visualization

param_grid_long <- param_grid |>
  tidyr::pivot_longer(
    cols = -run,
    names_to = "parameter",
    values_to = "param_value"
  )

# Join results with parameter information
combined_data <- readr::read_csv(paste0(output_base_path, "_combined_biomass_succession_data.csv"))

plot_data <- combined_data |>
  dplyr::left_join(param_grid_long, by = "run", relationship = "many-to-many")

This first graph shows all the model runs for the parameter sweep, faceted by each parameter. Here you can see how each parameter influences the shape of the trajectory. 

In [ ]:
library(ggplot2)

run_trajectories <- plot_data |>
  ggplot(aes(x = Time, y = AvgLiveB, colour = param_value, group = run)) +
  geom_line(
    aes(colour = param_value), 
    linewidth = 0.3,
    alpha = 0.6
  ) +
  facet_wrap(~parameter, nrow = 2, ncol = 3, scales = "free_y") +
  scale_colour_manual(
    values = c("initial" = "#DC267F", "min" = "#648FFF", "max" = "#FFB000"),
    name = "Parameter Value"
  ) +
  theme_minimal() +
  theme(
    strip.text = element_text(face = "bold", size = 12),
    legend.position = "bottom",
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    axis.line = element_line(linewidth = 0.5, colour = "black"),
    axis.ticks = element_line(linewidth = 0.1, colour = "black")
  ) +
  labs(
    x = "Time",
    y = "Aboveground Biomass (g/m²)",
  )

print(run_trajectories)

ggsave(paste0(output_base_path, "_loobos_model_run_trajectories.png"))

This second graph shows the influence of each parameter and the parameter value when all other values are set to initial. Here you can tease apart the influence of the minimum and maximum values without all the noise. Code is adjustable that you can replace "initial" with "min" or "max" and see how the different parameter values influence the outcome depending on if all other runs are set to initial, min or max. 

In [ ]:

all_params_data <- dplyr::bind_rows(
  combined_data |> dplyr::left_join(param_grid, by = "run") |> 
    dplyr::filter(
        BiomassMax == "initial", 
        GrowthCurve == "initial", 
        MortalityCurve == "initial", 
        Longevity == "initial", 
        ProbEstablish == "initial") |>
    tidyr::pivot_longer(
        cols = ANPPMax, 
        names_to = "parameter", 
        values_to = "param_value"),
  
  combined_data |> dplyr::left_join(param_grid, by = "run") |> 
    dplyr::filter(
        ANPPMax == "initial", 
        GrowthCurve == "initial", 
        MortalityCurve == "initial", 
        Longevity == "initial", 
        ProbEstablish == "initial") |>
    tidyr::pivot_longer(
        cols = BiomassMax, 
        names_to = "parameter", 
        values_to = "param_value"),
  
  combined_data |> dplyr::left_join(param_grid, by = "run") |> 
    dplyr::filter(
        BiomassMax == "initial", 
        ANPPMax == "initial", 
        MortalityCurve == "initial", 
        Longevity == "initial", 
        ProbEstablish == "initial") |>
    tidyr::pivot_longer(
        cols = GrowthCurve, 
        names_to = "parameter", 
        values_to = "param_value"),
  
  combined_data |> dplyr::left_join(param_grid, by = "run") |> 
    dplyr::filter(
        BiomassMax == "initial", 
        GrowthCurve == "initial", 
        ANPPMax == "initial", 
        Longevity == "initial", 
        ProbEstablish == "initial") |>
    tidyr::pivot_longer(
        cols = MortalityCurve, 
        names_to = "parameter", 
        values_to = "param_value"),
  
  combined_data |> dplyr::left_join(param_grid, by = "run") |> 
    dplyr::filter(
        BiomassMax == "initial", 
        GrowthCurve == "initial", 
        MortalityCurve == "initial", 
        ANPPMax == "initial", 
        ProbEstablish == "initial") |>
    tidyr::pivot_longer(
        cols = Longevity, 
        names_to = "parameter", 
        values_to = "param_value"),
  
  combined_data |> dplyr::left_join(param_grid, by = "run") |> 
    dplyr::filter(
        BiomassMax == "initial", 
        GrowthCurve == "initial", 
        MortalityCurve == "initial", 
        Longevity == "initial", 
        ANPPMax == "initial") |>
    tidyr::pivot_longer(
        cols = ProbEstablish, 
        names_to = "parameter", 
        values_to = "param_value")
)

# Background: all runs
background_all <- combined_data |> 
  dplyr::left_join(param_grid, by = "run")

# Create single faceted plot
fig_params <- ggplot() +
  geom_line(
    data = background_all,
    aes(x = Time, y = AvgLiveB, group = run),
    color = "#CCCCCC",
    alpha = 0.15,
    linewidth = 0.3
  ) +
  geom_line(
    data = all_params_data,
    aes(x = Time, y = AvgLiveB, color = param_value, group = run),
    alpha = 0.9,
    linewidth = 0.8
  ) +
  facet_wrap(~parameter, nrow = 2, ncol = 3, scales = "free_y",
             labeller = labeller(parameter = c(
               ANPPMax = "ANPPMax",
               BiomassMax = "BiomassMax",
               GrowthCurve = "GrowthCurve",
               MortalityCurve = "MortalityCurve",
               Longevity = "Longevity",
               ProbEstablish = "ProbEstablish"
             ))) +
  scale_color_manual(
    values = c("initial" = "#DC267F", "min" = "#648FFF", "max" = "#FFB000"),
    name = "Parameter Value",
    guide = guide_legend(nrow = 1, title.position = "top", title.hjust = 0.5)
  ) +
  theme_minimal() +
  theme(
    strip.text = element_text(face = "bold", size = 10),
    plot.title = element_text(size = 12, face = "bold", margin = margin(b = 12)),
    axis.title = element_text(size = 10, face = "bold"),
    axis.text = element_text(size = 10, face = "bold"),
    legend.position = "bottom",
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    axis.line = element_line(linewidth = 0.3, colour = "black"),
    axis.ticks = element_line(linewidth = 0.3, colour = "black"),
    plot.margin = margin(t = 6, r = 8, b = 6, l = 6, unit = "pt")
  ) +
  labs(
    x = "Year",
    y = "Aboveground Biomass (g/m²)"
  )

print(fig_params)

ggsave(paste0(output_base_path, "_loobos_parameter_influence.png"))

# Calibration phase

The model runs are compared to the inventory data for Loobos which is aggregated into one file with the biomass calculated in the same way as in the model. 

In [ ]:
# Loobos inventory data cleaner
# ========================================

data1996 <- readr::read_csv(paste0(conf_data_input_location, "Loobos1996.csv")) |>
  dplyr::select(oldID, dbh_cm) |>
  dplyr::mutate(Time = 36) |>
  na.omit(dbh_cm)

data2000 <- readr::read_csv(paste0(conf_data_input_location, "Loobos2000.csv")) |>
  dplyr::select(oldID, dbh_cm) |>
  dplyr::mutate(Time = 40) |>
  na.omit(dbh_cm)

data2005 <- readr::read_csv(paste0(conf_data_input_location, "Loobos2005.csv")) |>
  dplyr::select(oldID, dbh_cm) |>
  dplyr::mutate(Time = 45) |>
  na.omit(dbh_cm)

data2008 <- readr::read_csv(paste0(conf_data_input_location, "Loobos2008.csv")) |>
  dplyr::select(oldID, dbh_cm)|>
  dplyr::mutate(Time = 48) |>
  na.omit(dbh_cm)

data2012 <- readr::read_csv(paste0(conf_data_input_location, "Loobos2012.csv")) |>
  dplyr::select(oldID, dbh_cm) |>
  dplyr::mutate(Time = 52) |>
  na.omit(dbh_cm)

data2025 <- readr::read_csv(paste0(conf_data_input_location, "Loobos2025.csv")) |>
  dplyr::select(oldID, dbh_cm) |>
  dplyr::mutate(Time = 65) |>
  na.omit(dbh_cm)

testing_data <- dplyr::bind_rows(data1996, data2000, data2005, data2008, data2012, data2025) # Excluding 2023 which will be testing independently

In [ ]:
# Biomass calculations
# ===============================================
# e^(log(c0) + c1 * log(dbh_cm)) * cf
# values from Forrester et al. 2017 https://doi.org/10.1016/j.foreco.2017.04.011

b0 <- 13.79
b1 <- -5.92
b2 <- 0.85

# Calculate volume
testing_data_bio <- testing_data |>
  dplyr::mutate(
    "volume_dm3" = b0 + b1 * dbh_cm + b2 * dbh_cm^2,
    "volume_m3" = volume_dm3 / 1000
  ) |>
#Calculate biomass using volume
  dplyr::mutate(
    "stem_biomass_kg" = volume_m3 * 420 #wood density in ton DM/m3 converted to kg/m3
  ) |>
  dplyr::mutate(
    "branch_biomass_kg" = exp(1)^(-3.6641 + 2.1601 * log(dbh_cm)) * 1.0451
  ) |>
  dplyr::mutate(
    "leaf_biomass_kg" =  exp(1)^(-3.5276 + 1.7471 * log(dbh_cm)) * 1.0104
  ) |>
# Calculate total aboveground biomass
  dplyr::mutate(
    "total_abg_biomass_kg" = stem_biomass_kg + branch_biomass_kg + leaf_biomass_kg, 
    "total_abg_bio_g" = total_abg_biomass_kg * 1000 #gives aboveground biomass in grams, for each individual tree which we can use for summary statistics
  ) |>
  dplyr::mutate(biomass_kg_ha = total_abg_biomass_kg* 445) |> #as if each tree represents a forest here we can see maximum and minimum potential ranges
# Convert to correct units
  dplyr::mutate(biomass_g_m = (biomass_kg_ha * 1000) / 10000) |>
#Select only necessary columns
  dplyr::select(oldID, Time, total_abg_bio_g, biomass_g_m)

readr::write_csv(testing_data_bio, paste0(output_base_path, "_observed_biomass.csv"))

## Basic statistics & summaries

In [ ]:
# Simple summary statistics
# =============================

# Simulated data
combined_data <- readr::read_csv(paste0(output_base_path, "_combined_biomass_succession_data.csv"))

# Summary by time step - across all runs but separated by timestep.
combined_data_summary <- combined_data |>
  dplyr::group_by(Time) |>
  dplyr::summarise(
    mean_AvgLiveB = mean(AvgLiveB, na.rm = TRUE),
    sd_AvgLiveB = sd(AvgLiveB, na.rm = TRUE),
    var_AvgLiveB = var(AvgLiveB, na.rm = TRUE),
    med_AvgLiveB = median(AvgLiveB, na.rm = TRUE),
    range_AvgLiveB = max(AvgLiveB) - min(AvgLiveB)
  )

combined_data_summary

# Observed data
observed_data <- readr::read_csv(paste0(output_base_path, "_observed_biomass.csv"))

#Summary by time step
combined_data_summary_obs <- observed_data |>
  dplyr::group_by(Time) |>
  dplyr::summarise(
      mean_biomass = round(mean(biomass_g_m, na.rm = TRUE)),
      sd_biomass = round(sd(biomass_g_m, na.rm = TRUE)),
      var_biomass = round(var(biomass_g_m, na.rm = TRUE)),
      med_biomass = round(median(biomass_g_m, na.rm = TRUE)),
      range_biomass = round(max(biomass_g_m) - min(biomass_g_m))
  )

combined_data_summary_obs

The summary statistics above show the average (mean) and median biomass values for each time step. For the simulated values you can see the average AvLiveB across all runs for each year of the model, as well as the standard deviation, variance and range. Likewise, for each year the average, standard deviation and variance for all the loobos trees are indicated according to year. 

In [ ]:
#Compare observed vs simualted for inventory years

# Observed values grouped according to time
observed_values <- combined_data_summary_obs |>
  dplyr::filter(Time %in% c(36, 40, 45, 48, 52, 65)) |>
  dplyr::select(Time, mean_biomass, med_biomass, sd_biomass) |>
  dplyr::rename(
      Time = Time,
      biomass_obs = mean_biomass,
      sd_obs = sd_biomass,
      med_biomass_obs = med_biomass
  ) |>
  dplyr::arrange(Time)

  # Simulated values for years 36, 40, 45, 48, 52, 65
simulated_values <- combined_data |>
  dplyr::filter(Time %in% c(36, 40, 45, 48, 52, 65)) |>
  dplyr::select(Time, AvgLiveB, run) |>
  dplyr::rename(
      Time = Time,
      biomass_sim = AvgLiveB,
      run = run
  )

# join the two - work with one file going forward
paired <- simulated_values |>
  dplyr::left_join(observed_values, by = "Time") |>
  dplyr::mutate(
      diff = biomass_sim - biomass_obs
  )

The paired values show us the average (mean) biomass values for the years against which we will compare, those matching the inventory years. 

In [ ]:
# Observed growth trajectories for specific trees
paired_trees <- intersect(intersect(intersect(intersect(intersect(data1996$oldID, data2000$oldID), data2005$oldID), data2008$oldID), data2012$oldID), data2025$oldID)

testing_data_obs <- testing_data_bio |>
  dplyr::mutate(tree_type = dplyr::if_else(oldID %in% paired_trees, "paired", "unpaired")) |>
  dplyr::filter(tree_type == "paired") |>
  dplyr::group_by(oldID)

ggplot(data = testing_data_obs, mapping = aes(x = Time, y = biomass_g_m)) +
  geom_point(aes(colour = oldID)) +
  geom_line(aes(colour=oldID)) +
  labs(x = "Time (years 1996 - 2025)",
  y = "Biomass (g/m²)") +
  theme_classic() +
  theme(legend.position = "none")

# Overlay these trajectories on top of all model runs
ggplot() +
  # Simulated runs (background, low emphasis)
  geom_line(data = combined_data, 
            aes(x = Time, y = AvgLiveB, group = run),
            colour = "gray", alpha = 0.2, linewidth = 0.4) +
  # Observed trajectories (foreground, high emphasis)
  geom_line(data = testing_data_obs, 
            aes(x = Time, y = biomass_g_m, colour = oldID, group = oldID),
            linewidth = 0.4, alpha = 0.8) +
  geom_point(data = testing_data_obs,
             aes(x = Time, y = biomass_g_m, colour = oldID),
             size = 1) +
  xlim(0, 100) +
  labs(x = "Time (years)",
       y = "Biomass (g/m²)",
       colour = "Observed Tree ID") +
  theme_classic() +
  theme(legend.position = "none")

The above cell takes the tree (oldID) that is present from 1996 all the way to 2025. If each tree represented a hypothetical forest we can obtain an understanding of the range of potential biomass the Loobos forest could obtain. This is shown here. 

In [ ]:
ggplot() +
  # Simulated runs (background, low emphasis)
  geom_line(data = combined_data, 
            mapping = aes(x = Time, y = AvgLiveB, group = run),
            colour = "gray", alpha = 0.2, linewidth = 0.4) +
# Observed values
  geom_point(
      data = testing_data_bio,
      aes(x = Time, y = biomass_g_m, group = biomass_g_m, colour = "Observed data"),
      alpha = 0.3, size = 0.3
  ) +
  # Observed uncertainty (ribbon)
  geom_ribbon(data = combined_data_summary_obs,
              mapping = aes(
                  x = Time, 
                  ymin = mean_biomass - sd_biomass,
                  ymax = mean_biomass + sd_biomass,
                  fill = "Observed uncertainty (±1 SD)"),
              alpha = 0.2) +
  # Average biomass for all trees in loobos
  geom_line(
      data = combined_data_summary_obs,
      aes(x = Time, y = mean_biomass, colour = "Mean biomass"),
      alpha = 0.5, linewidth = 0.8
  ) +
  geom_line(
      data = combined_data_summary_obs,
      aes(x = Time, y = med_biomass, colour = "Median biomass"),
      alpha = 0.5, linewidth = 0.8
  ) +
  scale_color_manual(
    values = c("Observed data" = "black", 
               "Mean biomass" = "blue",
               "Median biomass" = "purple"),
    name = ""
  ) +
  scale_fill_manual(
    values = c("Observed uncertainty (±1 SD)" = "lightblue"),
    name = ""
  ) +
  labs(x = "Time (years)",
       y = "Aboveground Biomass (g/m²)") +
  theme_minimal() +
  theme(
    axis.title = element_text(size = 10, face = "bold"),
    axis.text = element_text(size = 10, face = "bold"),
    legend.position = "bottom",
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    axis.line = element_line(linewidth = 0.3, colour = "black"),
    axis.ticks = element_line(linewidth = 0.3, colour = "black"),
    plot.margin = margin(t = 6, r = 8, b = 6, l = 6, unit = "pt")
  )

ggsave(paste0(output_base_path, "_average_loobos_trajectories.png"))

The above cell shows the average and median biomass observed across the Loobos inventories. Additionally, it visualises where these trajectories lie in relation to the simulated values. Finally, the points show us the amount of variation being represented by the mean.  

## Goodness of fit analysis

In [ ]:
library(hydroGOF)

A series of goodness-of-fit measures have been selected for the calibration of the model. 

In [ ]:
# GOF per run, pooling across all validation years 
gof_by_run <- paired |>
  dplyr::group_by(run) |>
  dplyr::group_modify(~ {
    g <- gof(sim = .x$biomass_sim, obs = .x$biomass_obs)
    dplyr::as_tibble(t(g), rownames = NULL)
  }) |>
  dplyr::ungroup()

samp_gof <- gof_by_run |>
  dplyr::select(
    c(run, ME, MAE, MSE, RMSE, NSE, r, R2, KGE, d)
  )

gof_by_run_med <- paired |>
  dplyr::group_by(run) |>
  dplyr::group_modify(~ {
      g <- gof(sim = .x$biomass_sim, obs = .x$med_biomass_obs)
      dplyr::as_tibble(t(g), rownames = NULL)
  }) |>
  dplyr::ungroup()

med_gof <- gof_by_run_med |>
  dplyr::select(run, ME, MAE, MSE, RMSE, NSE, r, R2, KGE, d)

med_gof

samp_gof contains the outputs for:
* Mean error (ME)
* Mean absolute error (MAE)
* Mean square error (MSE)
* Root mean square error (RMSE)
* Correlation coefficient (r)
* Coefficient of determination (r2)
* Nash-Sutcliffe index of efficiency (NSE)
* Kling-Gupta index of efficiency (KGE)

### R2

Directly shows variance explained by the model. 
Range: -∞ to 1 (1 = perfect, 0 = as good as mean, <0 = worse than mean)

In [ ]:
R2_test <- paired |>
  dplyr::group_by(run) |>
  dplyr::summarise(
      R2 = hydroGOF::R2(biomass_sim, biomass_obs)
  ) |>
  dplyr::arrange(desc(R2)) # largest value first

print(R2_test, nrow = 10)

R2_test_med <- paired |>
  dplyr::group_by(run) |>
  dplyr::summarise(
      R2 = hydroGOF::R2(biomass_sim, med_biomass_obs)
  ) |>
  dplyr::arrange(desc(R2)) # largest value first

print(R2_test_med, nrow = 10)

The best run according to R2 is run 507 for mean and run 335 for median

### RMSE
Measure the average error provided in units that are the same as in the model

In [ ]:
RMSE_test <- paired |>
  dplyr::group_by(run) |>
  dplyr::summarise(
      RMSE = hydroGOF::ubRMSE(biomass_sim, biomass_obs)
  ) |>
  dplyr::arrange(RMSE) # smallest value first

print(RMSE_test, nrow = 10)

RMSE_test_med <- paired |>
  dplyr::group_by(run) |>
  dplyr::summarise(
      RMSE = hydroGOF::ubRMSE(biomass_sim, med_biomass_obs)
  ) |>
  dplyr::arrange(RMSE) # smallest value first

print(RMSE_test_med, nrow = 10)

The best run according to RMSE is 507

### NSE

Indicates whether or not the model is worse, as good as or better than using the mean to represent the system of interest. 

In [ ]:
NSE_test <- paired |>
  dplyr::group_by(run) |>
  dplyr::summarise(
      NSE = hydroGOF::NSE(biomass_sim, biomass_obs)
  ) |>
  dplyr::arrange(desc(NSE)) # largest value first

print(NSE_test, nrow = 10)

NSE_test_med <- paired |>
  dplyr::group_by(run) |>
  dplyr::summarise(
      NSE = hydroGOF::NSE(biomass_sim, med_biomass_obs)
  ) |>
  dplyr::arrange(desc(NSE)) # largest value first

print(NSE_test_med, nrow = 10)

According to NSE, run 507 is the best run for mean and run 335 is the best for median. 

### KGE

Combines correlation, bias and variance.


In [ ]:
KGE_test <- paired |>
  dplyr::group_by(run) |>
  dplyr::summarise(
      KGE = hydroGOF::KGE(biomass_sim, biomass_obs, method = "2021")
  ) |>
  dplyr::arrange(desc(KGE)) # largest value first

print(KGE_test, nrow = 10)

KGE_test_med <- paired |>
  dplyr::group_by(run) |>
  dplyr::summarise(
      KGE = hydroGOF::KGE(biomass_sim, med_biomass_obs, method = "2021")
  ) |>
  dplyr::arrange(desc(KGE)) # largest value first

print(KGE_test_med, nrow = 10)

According to KGE, run 507 is the best fitting for mean while 578 is for median

In [ ]:
# Pick run 507
single_run <- paired |> dplyr::filter(run == "507")
single_run_med <- paired |> dplyr::filter(run == "578")
single_run_med_other <- paired |> dplyr::filter(run == "335")

# Get full KGE result
kge_result <- hydroGOF::KGE(single_run$biomass_sim, single_run$biomass_obs, method = "2021", out.type = "full")
kge_med <- hydroGOF::KGE(single_run_med$biomass_sim, single_run_med$med_biomass_obs, method = "2021", out.type = "full")
kge_med_other <- hydroGOF::KGE(single_run_med_other$biomass_sim, single_run_med_other$med_biomass_obs, method = "2021", out.type = "full")

# Access components
print(kge_result)
print(kge_med)
print(kge_med_other)

The best run 507 is that where:

* ANPPMax is max
* BiomassMax is initial
* GrowthCurve is max
* MortalityCurve is initial
* Longevity is initial
* ProbEstablish is max

The best run 335 is that where:

* ANPPMax is min
* BiomassMax is initial
* GrowthCurve is min
* MortalityCurve is initial
* Longevity is min 
* ProbEstablish is min

The best run 578 is that where:

* ANPPMax is min
* BiomassMax is initial
* GrowthCurve is min
* MortalityCurve is initial
* Longevity is min
* ProbEstablish is max

In [ ]:
# Option 1: Highlight just the best run (507)
best_runs <- c("507", "335", "578")
colors_palette <- c("blue", "purple", "darkgreen")
names(colors_palette) <- best_runs

ggplot() +
  # All simulated runs (background, low emphasis)
  geom_line(data = combined_data, 
            mapping = aes(x = Time, y = AvgLiveB, group = run),
            colour = "gray", alpha = 0.2, linewidth = 0.4) +
  # Best run(s) highlighted
  geom_line(
      data = combined_data |> dplyr::filter(run %in% best_runs),
      aes(x = Time, y = AvgLiveB, colour = run),
      alpha = 0.9, linewidth = 1.2
  ) +
  # Observed summary
  geom_line(
      data = combined_data_summary_obs,
      aes(x = Time, y = mean_biomass, colour = "Mean biomass"),
      alpha = 0.5, linewidth = 0.8
  ) +
  geom_line(
      data = combined_data_summary_obs,
      aes(x = Time, y = med_biomass, colour = "Median biomass"),
      alpha = 0.5, linewidth = 0.8
  ) +
  scale_color_manual(
    values = c(colors_palette, 
               "Mean biomass" = "blue",
               "Median biomass" = "purple"),
    name = ""
  ) +
  labs(x = "Time (years)",
       y = "Aboveground Biomass (g/m²)") +
  theme_minimal() +
  theme(
    axis.title = element_text(size = 10, face = "bold"),
    axis.text = element_text(size = 10, face = "bold"),
    legend.position = "bottom",
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    axis.line = element_line(linewidth = 0.3, colour = "black"),
    axis.ticks = element_line(linewidth = 0.3, colour = "black"),
    plot.margin = margin(t = 6, r = 8, b = 6, l = 6, unit = "pt")
  )